<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Shopping queries with hard negatives

**[Shopping queries with hard negatives](https://learning.nextia-ai.com/courses/search/m09/shopping-queries-with-hard-negatives/)** · Search and Ranking: Find the Best Match · Case study 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** rank real shopping queries with four labels (exact, substitute, complement, irrelevant) in English and Spanish, compare BM25, dense and hybrid search, a cross-encoder and LambdaMART on quality, time and cost, and decide what to show when no exact product exists.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page; the notebook runs in about 1 minute |
| **Needs** | An internet connection: about 40 MB of data, and the bi-encoder (about 200 MB) in section 13. No account and no key. |
| **You should know** | The whole course; the lesson page links each lesson where it is used. |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, bm25s 0.3.16, PyStemmer 3.1.0, LightGBM 4.7.0, sentence-transformers 6.2.0 and PyTorch 2.15.0, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m09/shopping-queries-with-hard-negatives/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M09-L02-shopping-queries-with-hard-negatives/shopping-queries-with-hard-negatives-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses. It needs an internet connection and takes about a minute the first time (PyTorch is about 200 MB; Colab already has it).

You should see one line per package with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 bm25s==0.3.16 PyStemmer==3.1.0 lightgbm==4.7.0 sentence-transformers==6.2.0 torch==2.15.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'bm25s', 'PyStemmer', 'lightgbm', 'sentence-transformers', 'torch']:
    print(name, version(name))

## Setup: get the data

Run the next cell. It downloads the course team's sample of the Shopping Queries Dataset with the recorded model outputs (about 40 MB) and the course's helper `compare.py` into a folder `esci-search`, and checks each file's checksum. It needs no account. It takes about 20 seconds.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `esci-search`, then run the cell again.

**Source and licence.** Shopping Queries Dataset (ESCI): Chandan K. Reddy and others, "Shopping Queries Dataset: A Large-Scale ESCI Benchmark for Improving Product Search", 2022. [github.com/amazon-science/esci-data](https://github.com/amazon-science/esci-data), Apache-2.0, Copyright Amazon.com, Inc. or its affiliates. The course team took a sample (English and Spanish, 1,400 queries) and cut long product texts; the [dataset card](dataset.md) lists every change. The embeddings and the recorded cross-encoder scores were made by the course team with IBM's granite models (Apache-2.0).

In [ ]:
# Setup: download the files into esci-search/ and check each one.
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

FILES = {
    "data/esci_queries.csv": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L02-shopping-queries-with-hard-negatives/data/esci_queries.csv",
        "449213c0cf43043717ffc4e692ab0bb570db3e18d4c29bb4e0d55396eaf7cdce"
    ],
    "data/esci_judgments.csv.gz": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L02-shopping-queries-with-hard-negatives/data/esci_judgments.csv.gz",
        "2bc8ed25b3fb511e2d788570032dbe74bfeeead9dfba6ae3f82d316276af4ae1"
    ],
    "data/esci_products.csv.gz": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L02-shopping-queries-with-hard-negatives/data/esci_products.csv.gz",
        "2fa0682a01cff6d7ac44ef3a22a2b6ca1137df38bca825030482026e412d98b6"
    ],
    "data/esci_product_embeddings.npy": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L02-shopping-queries-with-hard-negatives/data/esci_product_embeddings.npy",
        "24e63d044de06b87bb9fa3c7800a136366bebf05fb581ea8e8766c598fef01c6"
    ],
    "data/esci_query_embeddings.npy": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L02-shopping-queries-with-hard-negatives/data/esci_query_embeddings.npy",
        "f37a40e0b52b217126bb81b4441cc5f57285480afd4efc40c982a9c36680d5ee"
    ],
    "data/esci_rerank_scores.csv.gz": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L02-shopping-queries-with-hard-negatives/data/esci_rerank_scores.csv.gz",
        "6e305db12eaa9237d82efc84a93ca851980cf7dbdfbecd065f7801fa0f21bad8"
    ],
    "data/esci_latency_ms.csv": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L02-shopping-queries-with-hard-negatives/data/esci_latency_ms.csv",
        "4ed83a25165b006f5aa29c766bdcfb9db2f0afee3fd415f3f8eb804e33c7d231"
    ],
    "data/model.json": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L02-shopping-queries-with-hard-negatives/data/model.json",
        "10b3be3327b4208bd79f7aa936a8076c9610e7c5cd5889a592d40590a51db168"
    ],
    "compare.py": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/compare.py",
        "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2"
    ]
}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():   # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("esci-search").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:   # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, (url, expected) in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name   # a copy next to the notebook, if there is one
        shutil.copy(local, path) if local.exists() else download(url, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder esci-search and run again.")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print(f"Ready: {len(FILES)} files in esci-search.")

## Setup: helpers

The next cell loads PyTorch first (on macOS this avoids a clash between PyTorch and LightGBM), NumPy, pandas, matplotlib and the course helper `compare.py` as `cp`. It also defines the check helpers `expect` and `expect_hash`: they tell you if your answer is right, without showing the answer. Run it. It prints nothing.

In [ ]:
# Helpers: load PyTorch first (macOS: avoids an OpenMP clash with LightGBM), then the rest.
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import torch
import json, re, time, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")
torch.set_num_threads(1)

def fingerprint(value):
    """A short hash of a result: the checks compare it without showing the answer."""
    import hashlib
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one of your numbers with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif isinstance(expected, float) and isinstance(yours, (int, float)) and abs(yours - expected) <= tolerance * max(1, abs(expected)):
        print(f"Check passed: {what} is {yours}.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

An online shop shows about 20 products for each search. Some are the product the person asked for. Others look alike but are not: a different model, an accessory, or a near miss. In this notebook you rank real shopping queries from Amazon's **Shopping Queries Dataset** (ESCI), in English and Spanish. Each candidate has one of four labels: **E**xact, **S**ubstitute, **C**omplement or **I**rrelevant. You compare BM25, dense search, hybrid search, a cross-encoder and LambdaMART, and you decide what to show when the exact product does not exist. The [lesson page](https://learning.nextia-ai.com/courses/search/m09/shopping-queries-with-hard-negatives/) tells the whole story, with the diagrams and the charts.

## 2. The data and the first checks

The sample has three files: queries, judgments (one ESCI label per query and product) and products. Run the next cell.

In [ ]:
queries = pd.read_csv("data/esci_queries.csv", keep_default_na=False)
labels = pd.read_csv("data/esci_judgments.csv.gz")
products = pd.read_csv("data/esci_products.csv.gz", keep_default_na=False)
print(f"{len(queries):,} queries, {len(labels):,} judgments, {len(products):,} products")
print(queries.groupby(["split", "locale"]).size().to_dict())
print(products.columns.tolist())

> **Check.** 1,400 queries (400 test and 300 train per language), 33,945 judgments and 32,759 products.

How common is each label? Run the next cell.

In [ ]:
lang = labels.merge(queries[["query_id", "locale"]], on="query_id")
shares = pd.crosstab(lang["locale"], lang["esci_label"], normalize="index")[["E", "S", "C", "I"]]
print(shares.round(3).to_string())

> **Check.** About 4 in 10 candidates are exact and about 1 in 3 are substitutes, in both languages. Spanish has more complements.

Before you measure anything, check the judgments: repeats, queries that cannot be scored, empty text.

In [ ]:
per_query = labels.groupby("query_id")
print("repeated (query, product) pairs:", int(labels.duplicated(["query_id", "product_id"]).sum()))
print("queries with no exact product:", int((~per_query["esci_label"].apply(lambda s: "E" in set(s))).sum()))
print("queries with only exact products:", int(per_query["esci_label"].apply(lambda s: set(s) == {"E"}).sum()))
print("judged products per query: median", int(per_query.size().median()), "| min", per_query.size().min(),
      "| max", per_query.size().max())
print("products without a title:", int((products["title"] == "").sum()))
print("products without bullet points:", int((products["bullets"] == "").sum()))
print("product IDs in both languages:", int(products["product_id"].duplicated().sum()))

In [ ]:
# A product is known by its language and its ID (23 IDs are in both languages, with different text).
products["key"] = products["locale"] + "/" + products["product_id"]
locale_of = dict(zip(queries["query_id"], queries["locale"]))
GRADE = {"E": 3, "S": 2, "C": 1, "I": 0}
judgments = {}
for q, p, g in zip(labels["query_id"], labels["product_id"], labels["esci_label"]):
    judgments.setdefault(int(q), {})[f"{locale_of[q]}/{p}"] = GRADE[g]
# The candidates of a query: every product that the shop's search showed and a person judged, in ID order.
CANDIDATES = {q: sorted(j) for q, j in judgments.items()}
test = queries[queries["split"] == "test"].reset_index(drop=True)
train = queries[queries["split"] == "train"].reset_index(drop=True)
print(f"test: {len(test)} queries, {sum(len(CANDIDATES[q]) for q in test['query_id']):,} candidates; "
      f"train: {len(train)} queries, {sum(len(CANDIDATES[q]) for q in train['query_id']):,} candidates")

## 3. Four labels, one measure

NDCG@10 needs a **gain** for each label. This notebook uses ESCI's own gains: exact 1, substitute 0.1, complement 0.01, irrelevant 0. The next cell gives those gains to `compare.py` and defines `measure`, which also counts the labels in the top 10 and the expected add-to-carts of the page (an assumption, explained on the lesson page).

In [ ]:
# ESCI's own gains (the task 1 code of the dataset): exact 1, substitute 0.1, complement 0.01, irrelevant 0.
GAIN = {3: 1.0, 2: 0.1, 1: 0.01, 0: 0.0}
cp.gain = lambda grade: GAIN[grade]   # compare.py's NDCG now uses these gains

# An assumption for the value of a page (the course's click model; a substitute behaves like a partial match):
# examine(k) = 1/k; a click when examined; add to cart after a click.
CLICK = {3: 0.65, 2: 0.25, 1: 0.10, 0: 0.05}
CART = {3: 0.30, 2: 0.08, 1: 0.05, 0: 0.01}
NAME = {3: "E", 2: "S", 1: "C", 0: "I"}

def measure(run):
    """One row per query: NDCG@10 (ESCI gains), MRR and recall (exact only), the label mix of the top 10,
    and the expected add-to-carts of the page under the assumption above."""
    rows = []
    for q, ranked in run.items():
        j = judgments[q]
        top = [j.get(p, 0) for p in ranked[:10]]
        row = {"query_id": q, "ndcg@10": cp.ndcg(ranked, j, 10), "mrr": cp.mrr(ranked, j, min_grade=3),
               "recall@100": cp.recall(ranked, j, 100, min_grade=3), "zero": float(not ranked),
               "judged@10": cp.judged_at(ranked, j, 10), "exact@1": float(bool(top) and top[0] == 3),
               "carts": sum(CLICK[g] * CART[g] / k for k, g in enumerate(top, 1))}
        for g, n in NAME.items():
            row[n] = top.count(g) / len(top) if top else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index("query_id")

# A tiny example: one query with four candidates, in two orders.
j = {"a": 3, "b": 2, "c": 1, "d": 0}
print("exact first:", round(cp.ndcg(["a", "b", "c", "d"], j), 3), "| substitute first:", round(cp.ndcg(["b", "a", "c", "d"], j), 3),
      "| irrelevant first:", round(cp.ndcg(["d", "a", "b", "c"], j), 3))

> **Check.** A substitute in first place costs 0.311 of NDCG; an irrelevant product costs only a little more. With these gains, a substitute is worth little.

> **Your turn.** What is the NDCG@10 of the order substitute, exact, irrelevant, complement? Use `cp.ndcg(ranked, judged)`.

In [ ]:
tiny_ndcg = ...   # NDCG@10 of the order substitute, exact, irrelevant, complement (ESCI gains), 3 decimals
print(tiny_ndcg)

In [ ]:
expect('tiny_ndcg', tiny_ndcg, 0.688)

## 4. Text into words, and the BM25 indexes

`tokens` splits the text; `norm` also removes accents and cuts each word to its stem with the stemmer of the query's language (English or Spanish). There is one set of indexes per language.

In [ ]:
import Stemmer
SPLIT = re.compile(r"[^\w]+|_")
STEM = {"us": Stemmer.Stemmer("english"), "es": Stemmer.Stemmer("spanish")}

def tokens(text):
    """Plain words: lower case, split on everything that is not a letter or a digit."""
    return [t for t in SPLIT.split(str(text).lower()) if t]

def fold(text):
    """Lower case, no accents: 'Batería Sin Cable' -> 'bateria sin cable'."""
    t = unicodedata.normalize("NFKD", str(text).lower())
    return "".join(c for c in t if not unicodedata.combining(c))

def norm(text, locale):
    """Normalised words, stemmed in the query's language: 'Fundas de Móvil' -> ['fund', 'de', 'movil']."""
    return STEM[locale].stemWords(tokens(fold(text)))

print(norm("Fundas de Móvil", "es"), norm("Wireless Earbuds Cases", "us"))

In [ ]:
import bm25s

products["brand_colour"] = products["brand"] + " " + products["colour"]
products["all"] = products["title"] + " " + products["bullets"] + " " + products["description"]
FIELDS = ["title", "brand_colour", "bullets", "description"]
BY_LOCALE = {loc: g.reset_index(drop=True) for loc, g in products.groupby("locale")}
ROW = {loc: pd.Series(np.arange(len(g)), index=g["key"]) for loc, g in BY_LOCALE.items()}

def build_index(texts, analyzer):
    """A BM25 index (Lucene form, k1 1.2, b 0.75) of a list of texts."""
    index = bm25s.BM25(k1=1.2, b=0.75, method="lucene")
    index.index([analyzer(t) for t in texts], show_progress=False)
    return index

def bm25_scores(index, words, locale, keys):
    """The BM25 score of the given products (keys) for a list of query words."""
    words = [w for w in words if w in index.vocab_dict]
    if not words:
        return np.zeros(len(keys))
    return index.get_scores(words)[ROW[locale][keys].to_numpy()]

t0 = time.time()
IDX = {}
for loc, g in BY_LOCALE.items():
    IDX[(loc, "all")] = build_index(g["all"], tokens)          # plain words, one field (the baseline)
    for field in FIELDS:
        IDX[(loc, field)] = build_index(g[field], lambda t, loc=loc: norm(t, loc))
print(f"Built {len(IDX)} BM25 indexes for {len(products):,} products in {time.time() - t0:.0f} s.")

In [ ]:
RESULTS = {}

def order(scores, keys):
    """The keys sorted by score, best first (ties: by key, so the order never depends on the label)."""
    return [k for _, k in sorted(zip(-np.asarray(scores, dtype=float), keys))]

def evaluate(name, scorer, qs=None, show=True):
    """Score every candidate of every test query with scorer(query_id, text, locale, keys) and keep the measures."""
    qs = test if qs is None else qs
    run = {}
    for q, text, loc in zip(qs["query_id"], qs["query"], qs["locale"]):
        keys = CANDIDATES[q]
        run[q] = order(scorer(q, text, loc, keys), keys)
    pq = measure(run)
    RESULTS[name] = {"run": run, "pq": pq}
    if show:
        s = cp.summary(pq)
        print(f"{name:20s} NDCG@10 {s['ndcg@10']:.3f} ({s['ndcg@10 low']:.3f} to {s['ndcg@10 high']:.3f})  "
              f"exact first {pq['exact@1'].mean():.1%}  S in top 10 {pq['S'].mean():.1%}")
    return pq

## 5. The floor and the baseline

Every candidate was shown by the shop's search, so even a **random order** is not bad. BM25 is the baseline that every other method must beat, on the same 800 test queries.

In [ ]:
# The floor: the candidates in a random order (the same seed every time).
rng = np.random.default_rng(0)
evaluate("random order", lambda q, t, loc, keys: rng.random(len(keys)))

# The baseline: BM25 on title, bullet points and description as one field, plain words.
def bm25(q, text, loc, keys):
    return bm25_scores(IDX[(loc, "all")], tokens(text), loc, keys)
evaluate("bm25", bm25);

> **Check.** Random order about 0.543; BM25 about 0.682.

Now BM25 per field, with stemming. The field weights are chosen on the **train** queries only.

In [ ]:
def field_scores(text, loc, keys):
    """BM25 of each field after normalising and stemming in the query's language: {field: scores}."""
    words = norm(text, loc)
    return {f: bm25_scores(IDX[(loc, f)], words, loc, keys) for f in FIELDS}

def weighted(weights):
    """A scorer that adds the field scores with the given weights."""
    def score(q, text, loc, keys):
        s = field_scores(text, loc, keys)
        return sum(w * s[f] for f, w in weights.items())
    return score

# The field weights, chosen on the train queries only (title 1; the other fields from a small grid).
grid = []
for wb in (0.25, 0.5, 1.0):
    for wl in (0.25, 0.5, 1.0):
        for wd in (0.0, 0.25, 0.5):
            w = {"title": 1.0, "brand_colour": wb, "bullets": wl, "description": wd}
            pq = evaluate("grid", weighted(w), qs=train, show=False)
            grid.append({**w, "train ndcg@10": pq["ndcg@10"].mean()})
del RESULTS["grid"]
grid = pd.DataFrame(grid).sort_values("train ndcg@10", ascending=False)
print(grid.head(3).round(3).to_string(index=False))
WEIGHTS = grid.iloc[0][FIELDS].to_dict()
bm25_norm = weighted(WEIGHTS)
evaluate("bm25-norm", bm25_norm);

> **Check.** About 0.687: close to the baseline. Stemming and fields help little when every candidate already shares words with the query.

## 6. Dense search and hybrid search

The course team encoded every product (title, brand, colour) and every query once with IBM's multilingual granite bi-encoder.

In [ ]:
P = np.load("data/esci_product_embeddings.npy").astype(np.float32)   # one row per product (file order)
QE = np.load("data/esci_query_embeddings.npy").astype(np.float32)    # one row per query (file order)
QV = dict(zip(queries["query_id"], QE))
PROW = pd.Series(np.arange(len(products)), index=products["key"])
MODEL = json.load(open("data/model.json"))
print(f"product embeddings {P.shape}, query embeddings {QE.shape}; model {MODEL['model']}")

def dense(q, text, loc, keys, vector=None):
    """Cosine similarity of the query and each product (the vectors have length 1: a dot product)."""
    vector = QV[q] if vector is None else vector
    return P[PROW[keys].to_numpy()] @ vector

evaluate("dense", dense);

In [ ]:
def rrf(lists, k=60):
    """Reciprocal rank fusion: each product gets the sum of 1 / (k + rank) over the lists."""
    total = {}
    for ranked in lists:
        for rank, p in enumerate(ranked, 1):
            total[p] = total.get(p, 0.0) + 1.0 / (k + rank)
    return total

def hybrid(q, text, loc, keys, vector=None):
    fused = rrf([order(bm25_norm(q, text, loc, keys), keys), order(dense(q, text, loc, keys, vector), keys)])
    return [fused[k] for k in keys]

evaluate("hybrid-rrf", hybrid);

> **Check.** Dense about 0.683; hybrid about 0.702.

## 7. A cross-encoder

The course team scored every test candidate once with IBM's granite reranker. This model is trained for **English**.

> **Predict.** In the course, this English model hurt French queries. Will it help or hurt the Spanish queries here?

In [ ]:
CE = pd.read_csv("data/esci_rerank_scores.csv.gz")   # recorded: every test candidate, scored once
CE_SCORE = {(q, f"{locale_of[q]}/{p}"): s for q, p, s in zip(CE["query_id"], CE["product_id"], CE["score"])}
print(f"{len(CE):,} recorded pairs for {CE['query_id'].nunique()} test queries; model {MODEL['cross_encoder']['model']}")

evaluate("rerank-ce", lambda q, t, loc, keys: [CE_SCORE[(q, k)] for k in keys]);

> **Your turn.** Measure the cross-encoder against BM25 on the Spanish test queries only.

In [ ]:
ids = None               # the query IDs of the Spanish test queries (locale "es")
spanish_ce_gain = ...    # rerank-ce minus bm25 on those queries (cp.paired(...)["difference"]), 3 decimals
print(spanish_ce_gain)

In [ ]:
expect('spanish_ce_gain', spanish_ce_gain, 0.038, tolerance=0.002)

## 8. LambdaMART on simple features

LambdaMART learns how to order the candidates from the labels of the 600 **train** queries. Each candidate is a row of 14 simple features: BM25 per field, the dense score, the RRF score, how many query words are in the title, whether the brand is in the query, and a few facts about the query. The candidates of a train query are the products that the shop's search showed: its **hard negatives**.

In [ ]:
import lightgbm as lgb

NEGATION = {"without", "no", "not", "sin", "non", "free"}
TITLE_WORDS = {k: set(norm(t, loc)) for k, t, loc in zip(products["key"], products["title"], products["locale"])}
BRAND = dict(zip(products["key"], products["brand"].map(fold)))
FEATURES = ["bm25_title", "bm25_brand_colour", "bm25_bullets", "bm25_description", "bm25_plain", "dense",
            "rrf", "title_coverage", "brand_in_query", "title_words", "query_words", "query_has_number",
            "query_has_negation", "spanish"]

def features(q, text, loc, keys, vector=None):
    """One row per candidate with the simple features that LambdaMART reads."""
    f = pd.DataFrame({"key": keys})
    for field, s in field_scores(text, loc, keys).items():
        f[f"bm25_{field}"] = s
    f["bm25_plain"] = bm25_scores(IDX[(loc, "all")], tokens(text), loc, keys)
    f["dense"] = dense(q, text, loc, keys, vector)
    f["rrf"] = hybrid(q, text, loc, keys, vector)
    words = set(norm(text, loc))
    f["title_coverage"] = [len(words & TITLE_WORDS[k]) / max(1, len(words)) for k in keys]
    folded = fold(text)
    f["brand_in_query"] = [float(len(BRAND[k]) > 1 and BRAND[k] in folded) for k in keys]
    f["title_words"] = [len(TITLE_WORDS[k]) for k in keys]
    f["query_words"] = len(tokens(text))
    f["query_has_number"] = float(any(ch.isdigit() for ch in text))
    f["query_has_negation"] = float(bool(NEGATION & set(tokens(text))))
    f["spanish"] = float(loc == "es")
    return f

def table_for(qs):
    parts = []
    for q, text, loc in zip(qs["query_id"], qs["query"], qs["locale"]):
        f = features(q, text, loc, CANDIDATES[q])
        f["query_id"] = q
        f["label"] = [judgments[q][k] for k in f["key"]]
        parts.append(f)
    return pd.concat(parts, ignore_index=True)

t0 = time.time()
train_table, test_table = table_for(train), table_for(test)
print(f"train: {len(train_table):,} rows, test: {len(test_table):,} rows, {len(FEATURES)} features, "
      f"in {time.time() - t0:.0f} s")

In [ ]:
PARAMS = {"objective": "lambdarank", "n_estimators": 200, "learning_rate": 0.05, "num_leaves": 15,
          "min_child_samples": 20, "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8,
          "label_gain": [GAIN[g] for g in range(4)], "random_state": 0, "n_jobs": 1, "verbose": -1}

def fit(rows, features=FEATURES):
    """LambdaMART on whole queries: rows of one query stay together."""
    rows = rows.sort_values("query_id", kind="stable")
    model = lgb.LGBMRanker(**PARAMS)
    model.fit(rows[features], rows["label"], group=rows.groupby("query_id", sort=True).size().to_numpy())
    return model

ltr_model = fit(train_table)
LTR_SCORE = dict(zip(zip(test_table["query_id"], test_table["key"]), ltr_model.predict(test_table[FEATURES])))
evaluate("ltr", lambda q, t, loc, keys: [LTR_SCORE[(q, k)] for k in keys])
importance = pd.Series(ltr_model.booster_.feature_importance("gain"), index=FEATURES)
print((100 * importance / importance.sum()).sort_values(ascending=False).head(5).round(1).to_dict())

> **Check.** About 0.707: the best so far.

> **Your turn (change one thing).** Train LambdaMART without the two dense features (`dense`, `rrf`). How much do they add?

In [ ]:
lexical_only = ...   # FEATURES without "dense" and "rrf"
no_dense_ndcg = ...  # train LambdaMART on train_table with those features; its test NDCG@10, 3 decimals
print(no_dense_ndcg)

In [ ]:
expect('no_dense_ndcg', no_dense_ndcg, 0.699, tolerance=0.003)

### A failure case: easy negatives

A common shortcut: train with random products as negatives, because they are easy to get. Run the next cell.

In [ ]:
# A mistake on purpose: train with easy negatives. Each train query keeps its exact products; its negatives are
# random products of other train queries (same language), not the look-alikes that the shop's search showed.
rng = np.random.default_rng(1)
pool = {loc: [k for q in g["query_id"] for k in CANDIDATES[q]] for loc, g in train.groupby("locale")}
parts = []
for q, text, loc in zip(train["query_id"], train["query"], train["locale"]):
    exact = [k for k in CANDIDATES[q] if judgments[q][k] == 3]
    mine = set(CANDIDATES[q])
    others = [k for k in rng.choice(pool[loc], 3 * len(CANDIDATES[q]), replace=False) if k not in mine]
    others = list(dict.fromkeys(others))[:len(CANDIDATES[q]) - len(exact)]
    f = features(q, text, loc, exact + others)
    f["query_id"], f["label"] = q, [3] * len(exact) + [0] * len(others)
    parts.append(f)
easy_table = pd.concat(parts, ignore_index=True)
easy_model = fit(easy_table)
EASY_SCORE = dict(zip(zip(test_table["query_id"], test_table["key"]), easy_model.predict(test_table[FEATURES])))
evaluate("ltr easy negatives", lambda q, t, loc, keys: [EASY_SCORE[(q, k)] for k in keys]);

> **Warning.** About 0.692: clearly below LambdaMART trained on the shop's own candidates (0.707), and no clear gain over BM25 (0.682). Random products are easy to tell apart from the query, so the model learns "shares words with the query = good". In the shop's list every product shares words with the query, so that rule cannot separate an exact product from its look-alikes.

## 9. One table: quality, time and cost

The time of a search is the sum of its stages, measured on 80 test queries on the course team's laptop (1 thread). The cost uses an assumed price of US$0.17 an hour for a 4-vCPU server.

In [ ]:
LAT = pd.read_csv("data/esci_latency_ms.csv").set_index("query_id")   # measured on the course team's laptop
# LambdaMART: time the features and the model here (the query vector comes from the encode stage).
ltr_ms = []
for q in LAT.index:
    row = test[test["query_id"] == q].iloc[0]
    t0 = time.perf_counter()
    f = features(q, row["query"], row["locale"], CANDIDATES[q])
    ltr_model.predict(f[FEATURES])
    ltr_ms.append(1000 * (time.perf_counter() - t0))
LAT["ltr"] = ltr_ms
print(f"candidates per query: median {LAT['candidates'].median():.0f}")
print(LAT[["bm25", "encode", "dense", "ce", "ltr"]].median().round(2).to_dict())

In [ ]:
PRICE = 0.17   # US$ an hour for a 4-vCPU server (an assumption: check your provider's price)
DESIGNS = {    # run name: (what it is, the stages it runs)
    "random order": ("the floor: no ranking", []),
    "bm25": ("baseline: BM25, one field, plain words", ["bm25"]),
    "bm25-norm": ("BM25 per field, stemmed per language", ["bm25"]),
    "dense": ("bi-encoder cosine", ["encode", "dense"]),
    "hybrid-rrf": ("RRF of bm25-norm and dense", ["bm25", "encode", "dense"]),
    "rerank-ce": ("cross-encoder (English model) on every candidate", ["encode", "ce"]),
    "ltr": ("LambdaMART, 14 simple features", ["encode", "ltr"]),
}
comparison = cp.Table()
for name, (what, stages) in DESIGNS.items():
    ms = LAT[stages].sum(axis=1).to_numpy() if stages else np.zeros(len(LAT))
    comparison.add(name, what, RESULTS[name]["pq"], ms, PRICE)
frame = comparison.frame().drop(columns=["setting", "recall@100", "zero-result rate"])
frame["exact first"] = [RESULTS[n]["pq"]["exact@1"].mean() for n in DESIGNS]
frame["S in top 10"] = [RESULTS[n]["pq"]["S"].mean() for n in DESIGNS]
frame["C in top 10"] = [RESULTS[n]["pq"]["C"].mean() for n in DESIGNS]
print(frame.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

In [ ]:
base = RESULTS["bm25"]["pq"]
for name in ["bm25-norm", "dense", "hybrid-rrf", "rerank-ce", "ltr"]:
    d = cp.paired(base, RESULTS[name]["pq"])
    print(f"{name:12s} {d['difference']:+.3f} ({d['low']:+.3f} to {d['high']:+.3f}); "
          f"better on {d['better']}, worse on {d['worse']}, same on {d['same']}")
d = cp.paired(RESULTS["hybrid-rrf"]["pq"], RESULTS["ltr"]["pq"])
print(f"ltr against hybrid-rrf: {d['difference']:+.3f} ({d['low']:+.3f} to {d['high']:+.3f})")

In [ ]:
order_ = ["bm25", "bm25-norm", "dense", "hybrid-rrf", "rerank-ce", "ltr"]
mix = pd.DataFrame({n: [RESULTS[n]["pq"][c].mean() for c in "ESCI"] for n in order_}, index=list("ESCI")).T
ax = mix.plot.barh(stacked=True, figsize=(7, 3.2), color=["#1f5fae", "#f28e2b", "#76b7b2", "#bbbbbb"])
ax.set_xlabel("share of the top 10 (test queries)"); ax.invert_yaxis(); ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.0))
plt.tight_layout(); plt.show()

Your computer is not the course team's laptop. Time one stage yourself.

In [ ]:
# Your own computer: time the LambdaMART features and model on the same 80 queries.
mine = []
for q in LAT.index:
    row = test[test["query_id"] == q].iloc[0]
    t0 = time.perf_counter()
    ltr_model.predict(features(q, row["query"], row["locale"], CANDIDATES[q])[FEATURES])
    mine.append(1000 * (time.perf_counter() - t0))
print(f"LambdaMART, p50 ms: your computer {np.median(mine):.1f}, course laptop {LAT['ltr'].median():.1f}")

## 10. English and Spanish, and the worst slices

ESCI tags some queries by how they were made: **negations** ("without", "sin"), **parse_pattern** (queries built from patterns), **behavioral** (from shopping behaviour) and **other**.

In [ ]:
SLICE_RUNS = ["random order", "bm25", "bm25-norm", "dense", "hybrid-rrf", "rerank-ce", "ltr"]
test["kind"] = test["source"].replace({"nlqec": "other"})   # ESCI's query sources; 2 "nlqec" queries join "other"

def slice_table(column, measure="ndcg@10"):
    """Mean of a measure for each run and each value of a test-query column."""
    cols = {}
    for name in SLICE_RUNS:
        s = cp.by_slice(RESULTS[name]["pq"], test, column, measure).set_index(column)
        cols[name] = s[measure]
    t = pd.DataFrame(cols)
    t.insert(0, "queries", s["queries"])
    return t

by_language = slice_table("locale")
by_kind = slice_table("kind")
print(by_language.round(3).to_string())
print(by_kind.round(3).to_string())

In [ ]:
for loc in ("us", "es"):
    ids = test.loc[test["locale"] == loc, "query_id"]
    for name in ["dense", "rerank-ce", "ltr"]:
        d = cp.paired(RESULTS["bm25"]["pq"].loc[ids], RESULTS[name]["pq"].loc[ids])
        print(f"{loc} {name:10s} - bm25: {d['difference']:+.3f} ({d['low']:+.3f} to {d['high']:+.3f})")

In [ ]:
def worst_slice(name, table):
    """The slice where a run gains least over BM25 (or loses most)."""
    gain = table[name] - table["bm25"]
    return gain.idxmin(), round(float(gain.min()), 3)
for name in SLICE_RUNS[2:]:
    print(f"{name:12s} worst language: {worst_slice(name, by_language)}; worst kind: {worst_slice(name, by_kind)}")

In [ ]:
title_of = dict(zip(products["key"], products["title"]))
def show(q, run, n=5):
    """The first n results of a test query in a run, with their ESCI label."""
    text = test.set_index("query_id").loc[q, "query"]
    print(f'"{text}"  ({run})')
    for i, k in enumerate(RESULTS[run]["run"][q][:n], 1):
        print(f"  {i}. [{NAME[judgments[q][k]]}] {title_of[k][:70]}")

example = 1509   # "100w solar panel without controller", a negation query in English
for run in ("bm25", "rerank-ce", "ltr"):
    show(example, run)

## 11. What a substitute costs, and what to show without an exact product

Under the assumption of the click model, compare what each label earns in first place.

In [ ]:
# What one position is worth under the assumption (expected add-to-carts when shown first):
first = {NAME[g]: CLICK[g] * CART[g] for g in (3, 2, 1, 0)}
print("worth at position 1:", {k: round(v, 4) for k, v in first.items()})
print(f"a substitute earns {first['S'] / first['E']:.0%} of an exact product, "
      f"{first['S'] / first['I']:.0f} times an irrelevant one")
carts = pd.Series({n: RESULTS[n]["pq"]["carts"].mean() * 1000 for n in SLICE_RUNS})
print("expected add-to-carts per 1,000 searches:", carts.round(1).to_dict())

Now a constructed case: the shop does not sell the exact product. The next cell hides the exact candidates and ranks what is left.

In [ ]:
# What if the shop does not sell the exact product? Hide the exact candidates of every test query, then rank
# what is left with each method (the models are not retrained). A constructed case, not data from Amazon.
saved, saved_j = dict(CANDIDATES), judgments
CANDIDATES = {q: [k for k in keys if judgments[q][k] != 3] for q, keys in saved.items()}
judgments = {q: {k: g for k, g in j.items() if g != 3} for q, j in saved_j.items()}   # the ideal list changes too
no_exact = test[[any(judgments[q][k] > 0 for k in CANDIDATES[q]) for q in test["query_id"]]]
NOEX = {}
for name, scorer in [("random order", lambda q, t, loc, keys: np.random.default_rng(q).random(len(keys))),
                     ("bm25", bm25), ("dense", dense), ("hybrid-rrf", hybrid),
                     ("rerank-ce", lambda q, t, loc, keys: [CE_SCORE[(q, k)] for k in keys]),
                     ("ltr", lambda q, t, loc, keys: [LTR_SCORE[(q, k)] for k in keys])]:
    pq = evaluate(f"{name} (no exact)", scorer, qs=no_exact, show=False)
    run = RESULTS[f"{name} (no exact)"]["run"]
    NOEX[name] = {"ndcg@10": pq["ndcg@10"].mean(),
                  "S first": np.mean([judgments[q][run[q][0]] == 2 for q in no_exact["query_id"]]),
                  "I first": np.mean([judgments[q][run[q][0]] == 0 for q in no_exact["query_id"]]),
                  "carts per 1,000": 1000 * pq["carts"].mean()}
CANDIDATES, judgments = saved, saved_j
noex = pd.DataFrame(NOEX).T
print(f"{len(no_exact)} test queries keep at least one substitute or complement")
print(noex.round(3).to_string())

> **Check.** 740 test queries keep a substitute or a complement. Read the column "S first": how often each method shows a substitute first, and "I first": an irrelevant product first.

### Change one thing: the gains

Do the conclusions depend on ESCI's gains? Measure the same runs with the course's gains (substitute and complement as partial matches).

In [ ]:
# Change one thing: the gains. The course's gains (2**grade - 1) with substitutes and complements as partial
# matches: exact 3, substitute 1, complement 1, irrelevant 0.
COURSE_GAIN = {3: 3.0, 2: 1.0, 1: 1.0, 0: 0.0}
cp.gain = lambda grade: COURSE_GAIN[grade]
other = {n: np.nanmean([cp.ndcg(RESULTS[n]["run"][q], judgments[q]) for q in test["query_id"]])
         for n in SLICE_RUNS}
cp.gain = lambda grade: GAIN[grade]    # back to ESCI's gains
both = pd.DataFrame({"ESCI gains": {n: RESULTS[n]["pq"]["ndcg@10"].mean() for n in SLICE_RUNS},
                     "course gains": other})
print(both.round(3).to_string())

## 12. The decision under 200 ms

> **Your turn.** Which method has the best NDCG@10 and a p95 time within the budget? Run the next cell, then write the method's name.

In [ ]:
BUDGET = 200   # milliseconds for one search (the course's budget)
fits = frame[frame["p95 ms"] <= BUDGET].sort_values("ndcg@10", ascending=False)
print(fits[["method", "ndcg@10", "ndcg@10 95% CI", "p50 ms", "p95 ms", "US$ per 1,000 queries"]].to_string(index=False))

In [ ]:
choice = ...   # the method with the best NDCG@10 whose p95 time fits the budget (a string)
print(choice)

In [ ]:
expect_hash('choice', choice, 'e953b5c777bfb113')

## 13. End to end on new queries

A new query has no list from the shop's search. Here, a stand-in first stage (RRF of BM25 and dense search over every product of the language) finds 30 candidates, and LambdaMART orders them. The query vector is computed now with the same bi-encoder: the first run downloads it (about 200 MB). The sample has only 32,759 products, so a query finds good results only for kinds of products that are in it. Nobody judged these results: read them yourself.

In [ ]:
# The design for production: LambdaMART trained on all 600 train queries, on the shop's candidates.
encoder = None

def encode(text):
    """The query's embedding, computed now with the same bi-encoder (downloaded the first time, about 200 MB)."""
    global encoder
    if encoder is None:
        from sentence_transformers import SentenceTransformer
        encoder = SentenceTransformer(MODEL["model"], revision=MODEL["revision"], device="cpu",
                                      model_kwargs={"dtype": torch.float32})
        encoder.max_seq_length = MODEL["max_tokens"]
    return encoder.encode([text], normalize_embeddings=True, convert_to_numpy=True)[0].astype(np.float32)

def first_stage(text, loc, vector, n=30):
    """Stand-in for the shop's own search: the RRF of BM25 and dense search over every product of the language."""
    keys = BY_LOCALE[loc]["key"].tolist()
    lex = order(bm25_norm(None, text, loc, keys), keys)[:100]
    den = order(dense(None, text, loc, keys, vector), keys)[:100]
    fused = rrf([lex, den])
    return sorted(fused, key=lambda k: (-fused[k], k))[:n]

def search(text, loc, n=5):
    """The recommended design on any query text: candidates -> LambdaMART -> the first n products."""
    t0 = time.perf_counter()
    vector = encode(text)
    keys = first_stage(text, loc, vector)
    f = features(None, text, loc, keys, vector)
    f["score"] = ltr_model.predict(f[FEATURES])
    f = f.sort_values(["score", "key"], ascending=[False, True])
    return f["key"].head(n).tolist(), 1000 * (time.perf_counter() - t0)

In [ ]:
for text, loc in [("wireless earbuds for running", "us"), ("phone case without magnet", "us"),
                  ("auriculares inalambricos para correr", "es")]:
    found, ms = search(text, loc)
    print(f'"{text}" ({loc}, {ms:.0f} ms)')
    for i, k in enumerate(found, 1):
        print(f"  {i}. {title_of[k][:78]}")

> **Your turn.** Add a query of your own, in English (`"us"`) or Spanish (`"es"`), and run the cell again. Is the first result exact, a substitute, a complement or irrelevant?

## 14. Recap

- In a list of look-alikes, random order already scores 0.543. BM25 (0.682) is the baseline; LambdaMART on simple features (0.707) is the best design that fits the time budget.
- The cross-encoder is the best ranker here, in English and in Spanish, but it takes about 340 ms per search on one CPU thread. Easy negatives cost LambdaMART most of its gain.
- With ESCI's gains a substitute is worth one tenth of an exact product. When there is no exact product, show substitutes, labelled as such, before complements. The lesson page explains the recommendation, the limits, and how to make this a portfolio piece.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Search a help centre or a scientific corpus](https://learning.nextia-ai.com/courses/search/m09/search-a-help-centre-or-a-scientific-corpus/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m09/shopping-queries-with-hard-negatives/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The queries, products and labels are a sample of the Shopping Queries Dataset (ESCI) by Amazon (Apache-2.0; Copyright Amazon.com, Inc. or its affiliates). No customer data is in it.